In [11]:
import numpy as np
from sklearn.cluster import KMeans
import os

def iou(box, clusters):
    """
    Calculate the Intersection over Union (IoU) between a box and clusters.
    :param box: tuple or array, shifted to the origin (0, 0)
    :param clusters: numpy array of shape (k, 2) where k is the number of clusters
    :return: numpy array of shape (k, 0), the IoU values
    """
    x = np.minimum(clusters[:, 0], box[0])
    y = np.minimum(clusters[:, 1], box[1])
    
    if np.count_nonzero(x == 0) > 0 or np.count_nonzero(y == 0) > 0:
        raise ValueError("Box has no area")
    
    intersection = x * y
    box_area = box[0] * box[1]
    cluster_area = clusters[:, 0] * clusters[:, 1]
    
    iou_ = intersection / (box_area + cluster_area - intersection)
    
    return iou_

def avg_iou(boxes, clusters):
    return np.mean([np.max(iou(box, clusters)) for box in boxes])

def kmeans(boxes, k, dist=np.median):
    """
    Calculates k-means clustering with IoU metric.
    :param boxes: numpy array of shape (r, 2) where r is the number of boxes
    :param k: number of clusters
    :param dist: distance function
    :return: numpy array of shape (k, 2)
    """
    boxes = np.array(boxes)
    rows = boxes.shape[0]
    
    distances = np.empty((rows, k))
    last_clusters = np.zeros((rows,))
    
    np.random.seed()
    clusters = boxes[np.random.choice(rows, k, replace=False)]
    
    while True:
        for row in range(rows):
            distances[row] = 1 - iou(boxes[row], clusters)
        
        nearest_clusters = np.argmin(distances, axis=1)
        
        if (last_clusters == nearest_clusters).all():
            break
        
        for cluster in range(k):
            clusters[cluster] = dist(boxes[nearest_clusters == cluster], axis=0)
        
        last_clusters = nearest_clusters
    
    return clusters

def load_dataset(directory):
    """
    Load dataset with the bounding boxes from multiple annotation files.
    :param directory: path to the directory containing the annotation files
    :return: list of bounding boxes
    """
    dataset = []
    for filename in os.listdir(directory):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory, filename)
            with open(filepath) as f:
                for line in f:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    dataset.append([width, height])
    return np.array(dataset)

# Example usage:
# Load your dataset from a directory containing annotation files
# Each annotation file has lines in the format: class_id x_center y_center width height
dataset = load_dataset('./dataset/annotation')

# Normalize the boxes to the input size of the network (e.g., 416x416 for YOLOv3)
original_size = 1920
input_size = 1920
dataset = dataset / original_size * input_size

# Calculate the anchor boxes using k-means clustering
num_anchors = 200
anchors = kmeans(dataset, num_anchors)
anchors = anchors * original_size / input_size  # Scale the anchors back to the original size

print("Anchors: ", anchors)


Anchors:  [[149.04709748 149.04709748]
 [139.11062432 139.11062432]
 [240.7366701  240.7366701 ]
 [140.91566265 140.91566265]
 [ 72.03980354  72.03980354]
 [132.87376079 132.87376079]
 [210.22835634 210.22835634]
 [163.96670331 163.96670331]
 [161.74249875 161.74249875]
 [200.36107493 200.36107493]
 [187.16504854 187.16504854]
 [113.55478515 113.55478515]
 [106.54180167 106.54180167]
 [235.15416098 235.15416098]
 [ 59.97147957  59.97147957]
 [ 88.85859777  88.85859777]
 [205.17526624 205.17526624]
 [132.15631292 132.16019417]
 [ 87.32787297  87.32787297]
 [120.31757115 120.31757115]
 [192.51714006 192.51714006]
 [ 94.63157895  94.63157895]
 [134.91186072 134.91186072]
 [105.9890471  105.9890471 ]
 [137.37971873 137.37971873]
 [ 69.44721998  69.44721998]
 [201.861521   201.9047619 ]
 [140.26739427 140.26739427]
 [203.99587024 203.99587024]
 [ 98.33761446  98.33761446]
 [ 90.91223154  90.91223154]
 [ 73.54672992  73.54672992]
 [102.17696314 102.17696314]
 [228.4137931  228.4137931 ]
 [17